# 09 · TGSIM I-395 — HDV vs. AV style manifold, and pedestrians

**Part A (real data).** Speed profiles of human-driven vehicles (HDV) and
automated vehicles (AV) from the TGSIM I-395 dataset (`data/tgsim/`). Every agent
with more than 100 samples is resampled to a 200-point grid, decomposed with the
two-shot method, and the personalized components are clustered (k-means, k = 4).
The 2-D PCA "style manifold" shows where the AVs fall relative to the human
driver population.

**Part B (synthetic pedestrians).** A demonstration on synthetic pedestrian speed
profiles (walkers / joggers / stop-and-go) with k = 3. This part uses *generated*
data, exactly as in the paper; a small real pedestrian extract
(`data/tgsim/ped_traj_fb.csv`) is provided for users who want to try the same
pipeline on real trajectories.

**Figures produced:** `TGSIM_cluster.png`, `TGSIM_pedestrians.png`.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "bhd").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

from bhd import DATA_DIR, FIG_DIR, set_style

set_style(seaborn_white=True)
FONT_TITLE, FONT_LABEL, FONT_TICK, FONT_LEGEND = 24, 24, 20, 18

## Part A — HDV and AV speed profiles (real TGSIM data)

In [ ]:
files = {
    "HDV": DATA_DIR / "tgsim" / "hdv_traj_i395.csv",
    "AV": DATA_DIR / "tgsim" / "av_traj_i395.csv",
}

data_frames = []
for agent_type, filename in files.items():
    df = pd.read_csv(filename)
    if "speed_kf" not in df.columns:
        raise ValueError(f"'speed_kf' not found in {filename}")
    df["val"] = df["speed_kf"]
    df["type"] = agent_type
    df["unique_id"] = agent_type + "_" + df["id"].astype(str)
    data_frames.append(df[["unique_id", "val", "type"]])
full_df = pd.concat(data_frames, ignore_index=True)

# Keep only agents with enough samples
valid_counts = full_df["unique_id"].value_counts()
valid_ids = valid_counts[valid_counts > 100].index.tolist()

# Resample to a common grid (context alignment)
common_len = 200
matrix_data, meta_types = [], []
for uid in valid_ids:
    subset = full_df[full_df["unique_id"] == uid]
    val = subset["val"].values
    matrix_data.append(np.interp(np.linspace(0, 1, common_len), np.linspace(0, 1, len(val)), val))
    meta_types.append(subset["type"].iloc[0])

A = np.stack(matrix_data, axis=1)  # (time, vehicles)
types = np.array(meta_types)
type_counts = pd.Series(types).value_counts()
print(type_counts.to_dict())

In [ ]:
# Two-shot decomposition
pca_shared = PCA(n_components=1)
C = pca_shared.fit(A.T).components_.T
Psi = C.T @ A
A_shared = C @ Psi

personalized = []
for v in range(A.shape[1]):
    R_v = (A[:, v] - A_shared[:, v]).reshape(-1, 1)
    pca_personal = PCA(n_components=1)
    Phi_v = pca_personal.fit_transform(R_v)
    personalized.append(pca_personal.inverse_transform(Phi_v).flatten())
A_personal = np.column_stack(personalized)

# Clustering on the personalized curves
n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
labels = kmeans.fit_predict(A_personal.T)
cluster_colors = ["C0", "C1", "C2", "C3"]
print("Cluster sizes:", np.bincount(labels))
print("AV cluster assignments:", labels[types == "AV"])

In [ ]:
# Style manifold
pca_2d = PCA(n_components=2)
X_vis = pca_2d.fit_transform(A_personal.T)

plt.figure(figsize=(14, 11))
hdv_mask = types == "HDV"
plt.scatter(X_vis[hdv_mask, 0], X_vis[hdv_mask, 1], c=[cluster_colors[l] for l in labels[hdv_mask]], s=150, alpha=0.5, edgecolor="none")
av_mask = types == "AV"
plt.scatter(X_vis[av_mask, 0], X_vis[av_mask, 1], marker="*", s=1200, c=[cluster_colors[l] for l in labels[av_mask]], edgecolor="black", linewidth=3.0, zorder=100)

plt.xlabel("Personalized Component 1", fontsize=FONT_LABEL)
plt.ylabel("Personalized Component 2", fontsize=FONT_LABEL)
plt.xticks(fontsize=FONT_TICK)
plt.yticks(fontsize=FONT_TICK)
plt.grid(True, linestyle=":", alpha=0.6)

shape_handles = [
    Line2D([0], [0], marker="o", linestyle="None", markerfacecolor="gray", markeredgecolor="none", markersize=14, alpha=0.6, label=f'HDV (N={int(type_counts.get("HDV", 0))})'),
    Line2D([0], [0], marker="*", linestyle="None", markerfacecolor="white", markeredgecolor="black", markeredgewidth=2, markersize=16, label=f'AV (N={int(type_counts.get("AV", 0))})'),
]
leg1 = plt.legend(handles=shape_handles, loc="upper left", frameon=True, framealpha=0.95, fontsize=FONT_LEGEND, title="Agent Type", title_fontsize=FONT_LEGEND, borderpad=0.8, labelspacing=0.6, handletextpad=0.8)
cluster_handles = [Line2D([0], [0], marker="o", linestyle="None", markerfacecolor=cluster_colors[i], markeredgecolor="none", markersize=14, label=f"Cluster {i + 1}") for i in range(n_clusters)]
leg2 = plt.legend(handles=cluster_handles, loc="upper right", frameon=True, framealpha=0.95, fontsize=FONT_LEGEND, title="Speed Cluster", title_fontsize=FONT_LEGEND, borderpad=0.8, labelspacing=0.6, handletextpad=0.8)
plt.gca().add_artist(leg1)

plt.tight_layout()
plt.savefig(FIG_DIR / "TGSIM_cluster.png", dpi=300, bbox_inches="tight")
plt.show()

## Part B — Synthetic pedestrian speed profiles

> **Note:** this section generates synthetic pedestrian data (three behavior
> archetypes) rather than reading a trajectory file. It is a demonstration of the
> pipeline on a different agent class, as presented in the paper.

In [ ]:
def generate_pedestrian_data(n_agents=200, n_steps=200):
    """Synthetic pedestrian speed profiles: walkers, joggers, stop-and-go."""
    np.random.seed(42)
    data = []
    for i in range(n_agents):
        profile_type = np.random.choice(["Walker", "Jogger", "StopGo"], p=[0.6, 0.2, 0.2])
        if profile_type == "Walker":
            speed = np.abs(1.4 + np.random.normal(0, 0.2) + np.random.normal(0, 0.1, n_steps))
        elif profile_type == "Jogger":
            speed = np.abs(3.0 + np.random.normal(0, 0.5) + np.random.normal(0, 0.2, n_steps))
        else:
            speed = np.abs(np.sin(np.linspace(0, 10, n_steps)) * 1.5)
        for t, s in enumerate(speed):
            data.append({"id": i, "time_step": t, "speed_kf": s, "type": "Pedestrian"})
    return pd.DataFrame(data)


df = generate_pedestrian_data()
df["unique_id"] = df["type"] + "_" + df["id"].astype(str)
df["val"] = df["speed_kf"]

valid_counts = df["unique_id"].value_counts()
valid_ids = valid_counts[valid_counts > 100].index.tolist()

common_len = 200
matrix_data, meta_types = [], []
for uid in valid_ids:
    subset = df[df["unique_id"] == uid]
    val = subset["val"].values
    matrix_data.append(np.interp(np.linspace(0, 1, common_len), np.linspace(0, 1, len(val)), val))
    meta_types.append(subset["type"].iloc[0])

A = np.stack(matrix_data, axis=1)
types = np.array(meta_types)
type_counts = pd.Series(types).value_counts()

# Two-shot decomposition
pca_shared = PCA(n_components=1)
C = pca_shared.fit(A.T).components_.T
Psi = C.T @ A
A_shared = C @ Psi
personalized = []
for v in range(A.shape[1]):
    R_v = (A[:, v] - A_shared[:, v]).reshape(-1, 1)
    pca_personal = PCA(n_components=1)
    Phi_v = pca_personal.fit_transform(R_v)
    personalized.append(pca_personal.inverse_transform(Phi_v).flatten())
A_personal = np.column_stack(personalized)

n_clusters = 3
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
labels = kmeans.fit_predict(A_personal.T)
cluster_colors = ["C0", "C1", "C2", "C3"]
print("Cluster sizes:", np.bincount(labels))

In [ ]:
pca_2d = PCA(n_components=2)
X_vis = pca_2d.fit_transform(A_personal.T)

plt.figure(figsize=(14, 11))
ped_mask = types == "Pedestrian"
plt.scatter(X_vis[ped_mask, 0], X_vis[ped_mask, 1], c=[cluster_colors[l] for l in labels[ped_mask]], s=150, alpha=0.6, edgecolor="white", linewidth=0.5)
plt.xlabel("Personalized Component 1", fontsize=FONT_LABEL)
plt.ylabel("Personalized Component 2", fontsize=FONT_LABEL)
plt.xticks(fontsize=FONT_TICK)
plt.yticks(fontsize=FONT_TICK)
plt.grid(True, linestyle=":", alpha=0.6)

shape_handles = [Line2D([0], [0], marker="o", linestyle="None", markerfacecolor="gray", markeredgecolor="none", markersize=14, alpha=0.6, label=f'Pedestrian (N={int(type_counts.get("Pedestrian", 0))})')]
leg1 = plt.legend(handles=shape_handles, loc="upper left", frameon=True, framealpha=0.95, fontsize=FONT_LEGEND, title="Agent Type", title_fontsize=FONT_LEGEND, borderpad=0.8, labelspacing=0.6, handletextpad=0.8)
cluster_handles = [Line2D([0], [0], marker="o", linestyle="None", markerfacecolor=cluster_colors[i], markeredgecolor="none", markersize=14, label=f"Cluster {i + 1} (N={np.sum(labels == i)})") for i in range(n_clusters)]
leg2 = plt.legend(handles=cluster_handles, loc="upper right", frameon=True, framealpha=0.95, fontsize=FONT_LEGEND, title="Behavior Cluster", title_fontsize=FONT_LEGEND, borderpad=0.8, labelspacing=0.6, handletextpad=0.8)
plt.gca().add_artist(leg1)

plt.tight_layout()
plt.savefig(FIG_DIR / "TGSIM_pedestrians.png", dpi=300, bbox_inches="tight")
plt.show()